In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import nuclear
from einops import rearrange
from tqdm import tqdm

jax.config.update("jax_enable_x64", True)

N = 100
D = 6
G = 3

x = jr.normal(jr.key(0), (N, D))


# gemnerate some low rank matrices for each tasks
v1 = jr.normal(jr.key(1), (D, 1))
v2 = jr.normal(jr.key(2), (D, 1))
v3 = jr.normal(jr.key(3), (D, 1))
A1 = v1 @ v1.T + v2 @ v2.T
A2 = v1 @ v1.T + v3 @ v3.T
A3 = v2 @ v2.T + v3 @ v3.T

# generate the targets for each task
y = jnp.stack([x @ A1, x @ A2, x @ A3], axis=1)
y = y + 0.001 * jr.normal(jr.key(4), y.shape)

LAMBDAS = 10 ** jnp.linspace(0, 4, 20)#[::-1]
model = nuclear.LinearRegressor(x_train=x, y_train=y, max_iterations=1000, tollerance=1e-3)

r2s = []
nucs = []
ranks = []
for l in tqdm(LAMBDAS):
    model.fit(l1_penalty=l)
    y_pred = model.predict(x)
    r2 = 1 - jnp.mean((y - y_pred) ** 2, axis=0) / jnp.var(y, axis=0)
    nuc = jnp.linalg.norm(rearrange(model.parameters, "g i o -> (g i) o"), ord="nuc")
    sv = jnp.linalg.svdvals(rearrange(model.parameters, "g i o -> (g i) o"))
    rank = jnp.sum(sv > 1e-4)
    r2s.append(r2)
    nucs.append(nuc)
    ranks.append(rank)
    print(f"Lambda: {l:.2e}, R2: {r2.mean()}, Nuc: {nuc:.2e}, Rank: {rank}")
    print()

  5%|▌         | 1/20 [00:00<00:18,  1.03it/s]

Lambda: 1.00e+00, R2: 0.9999921055028071, Nuc: 1.52e+01, Rank: 6



ADMM:   0%|          | 3/1000 [00:00<00:03, 289.21it/s, primal:=0.00016, dual:=0.00034, rho=0.5]


Lambda: 1.62e+00, R2: 0.9999820002001064, Nuc: 1.52e+01, Rank: 6



ADMM:   0%|          | 3/1000 [00:00<00:03, 290.47it/s, primal:=0.00035, dual:=0.00013, rho=0.25]


...

Lambda: 3.79e+03, R2: -0.00206541443426632, Nuc: 2.91e-09, Rank: 0



 95%|█████████▌| 19/20 [00:01<00:00, 18.82it/s]

Lambda: 6.16e+03, R2: -0.00206541443426632, Nuc: 2.91e-09, Rank: 0



100%|██████████| 20/20 [00:01<00:00, 11.80it/s]

Lambda: 1.00e+04, R2: -0.00206541443426632, Nuc: 2.91e-09, Rank: 0

